# 1. Download the raw data

This notebook downloads the four indicators used in the project from [Our World in Data](https://ourworldindata.org/) and saves them **unchanged** in `data/raw/`.
The next notebook (`2_create_dataset.ipynb`) harmonizes them into one panel dataset.

> OWID URLs always serve the latest data, so re-running this notebook can give slightly different numbers. The committed `data/master_governance_dataset.csv` is the snapshot used by the app.

## Import the libraries

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd

## Get the data

In [ ]:
# Sources: Our World in Data (OWID)
urls = {
    "cpi": "https://ourworldindata.org/grapher/ti-corruption-perception-index.csv?v=1&csvType=full&useColumnShortNames=false",
    "pci": "https://ourworldindata.org/grapher/political-corruption-index.csv?v=1&csvType=full&useColumnShortNames=false",
    "gdp": "https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv?v=1&csvType=full&useColumnShortNames=false",
    "regime": "https://ourworldindata.org/explorers/democracy.csv?v=1&csvType=full&useColumnShortNames=false&Dataset=Regimes+of+the+World&Metric=%C2%ADPolitical+regime&Sub-metric=Main+classification",
}

HEADERS = {"User-Agent": "Our World In Data data fetch/1.0"}

In [ ]:
# Create the folder for raw data if it doesn't exist
output_dir = Path("../data/raw")
output_dir.mkdir(parents=True, exist_ok=True)

manifest = []
for name, url in urls.items():
    table = pd.read_csv(url, storage_options=HEADERS)

    # Save the untouched download locally
    file_path = output_dir / f"{name}_raw.csv"
    table.to_csv(file_path, index=False)

    manifest.append({
        "source": name,
        "rows": len(table),
        "countries": table["Entity"].nunique(),
        "year_min": table["Year"].min(),
        "year_max": table["Year"].max(),
        "downloaded_on": date.today().isoformat(),
        "url": url,
    })
    print(
        f"[{name.upper()}] saved to {file_path} | {len(table):,} rows, {table.shape[1]} cols | "
        f"{manifest[-1]['countries']} countries | Years: {manifest[-1]['year_min']}-{manifest[-1]['year_max']}"
    )

# OWID URLs always serve the latest data: keep a trace of what was downloaded, and when
pd.DataFrame(manifest).to_csv(output_dir / "download_manifest.csv", index=False)

### Variable scales & polarities (raw data)

Before any transformation, the raw indicators follow their original conventions:

* **CPI (Corruption Perceptions Index, Transparency International):** 0 to 100, **0 = highly corrupt**, **100 = very clean**.
* **PCI (Political Corruption Index, V-Dem):** 0 to 1, **0 = clean**, **1 = maximum political corruption** (opposite direction to CPI).
* **GDP_pc (GDP per capita, World Bank):** constant international dollars per capita (PPP).
* **Regime code (Regimes of the World, V-Dem):** `0 = Closed autocracy`, `1 = Electoral autocracy`, `2 = Electoral democracy`, `3 = Liberal democracy`.

The app aligns the polarities itself (corruption score = PCI x 100, perception score = 100 - CPI), so that on every chart higher means more corrupt.

### Source coverage worth remembering

* **CPI** starts in 2012 (Transparency International changed its method that year, earlier values are not comparable).
* The **download date** of every file is stored in `data/raw/download_manifest.csv`.